In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/complaint-sense-consumer-complaint-classification-challenge/sample_submission.csv
/kaggle/input/competitions/complaint-sense-consumer-complaint-classification-challenge/test_complaints.csv
/kaggle/input/competitions/complaint-sense-consumer-complaint-classification-challenge/dataset-metadata.json
/kaggle/input/competitions/complaint-sense-consumer-complaint-classification-challenge/train_complaints.csv
/kaggle/input/competitions/complaint-sense-consumer-complaint-classification-challenge/baseline_submission.csv


In [2]:
train_df = pd.read_csv("/kaggle/input/competitions/complaint-sense-consumer-complaint-classification-challenge/train_complaints.csv")
test_df = pd.read_csv("/kaggle/input/competitions/complaint-sense-consumer-complaint-classification-challenge/test_complaints.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("\nTrain columns:", list(train_df.columns))
print("Test columns:", list(test_df.columns))
train_df.head()

Train shape: (380, 4)
Test shape: (160, 2)

Train columns: ['ComplaintId', 'text', 'Category', 'FamilyId']
Test columns: ['ComplaintId', 'text']


,ComplaintId,text,Category,FamilyId
0,1,Flagging an issue: Username change broke acces...,account_access,template:account_access:5
1,2,Writing to complain: Blender motor smells like...,product_defect,template:product_defect:6
2,3,"Hello, Courier left perishable groceries in th...",delivery_shipping,template:delivery_shipping:1
3,4,Hi support — Tablet touchscreen registers ghos...,product_defect,template:product_defect:7
4,5,Hi support — Authorized service charged labor ...,warranty_repair,template:warranty_repair:3


In [3]:
import torch
import numpy as np
import pandas as pd
import random
import re
import unicodedata
import warnings

warnings.filterwarnings("ignore")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("""
    ❌ GPU NOT ENABLED

    In Kaggle:
    Settings -> Accelerator -> GPU T4 x2

    Then rerun the notebook.
    """)

print("\nTrain shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nColumns:")
print(train_df.columns)
print(test_df.columns)

print("\nClass distribution:")
print(train_df["Category"].value_counts())

CUDA available: True
GPU: Tesla T4

Train shape: (380, 4)
Test shape: (160, 2)

Columns:
Index(['ComplaintId', 'text', 'Category', 'FamilyId'], dtype='object')
Index(['ComplaintId', 'text'], dtype='object')

Class distribution:
Category
billing                50
account_access         40
refund_return          40
delivery_shipping      40
fraud_unauthorized     40
product_defect         35
general_inquiry        35
subscription_cancel    35
warranty_repair        35
customer_service       30
Name: count, dtype: int64


In [4]:
LABELS = [
    "billing",
    "product_defect",
    "delivery_shipping",
    "refund_return",
    "customer_service",
    "account_access",
    "fraud_unauthorized",
    "warranty_repair",
    "subscription_cancel",
    "general_inquiry",
]

label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for label, i in label2id.items()}

print(label2id)

# Make sure there are no unexpected labels
print("\nLabels actually present:")
print(sorted(train_df["Category"].unique()))

assert set(train_df["Category"].unique()).issubset(set(LABELS))

{'billing': 0, 'product_defect': 1, 'delivery_shipping': 2, 'refund_return': 3, 'customer_service': 4, 'account_access': 5, 'fraud_unauthorized': 6, 'warranty_repair': 7, 'subscription_cancel': 8, 'general_inquiry': 9}

Labels actually present:
['account_access', 'billing', 'customer_service', 'delivery_shipping', 'fraud_unauthorized', 'general_inquiry', 'product_defect', 'refund_return', 'subscription_cancel', 'warranty_repair']


In [5]:
# ============================================================
# SAFER CORE-TEXT NORMALIZATION
# ============================================================

PREFIX_PATTERNS = [

    # Support wrappers
    r"^\s*hi\s+support\s*[-–—,:;]+\s*",
    r"^\s*hello\s+support\s*[-–—,:;]+\s*",
    r"^\s*dear\s+support\s*[-–—,:;]+\s*",

    # Generic greeting wrappers
    r"^\s*hello\s*[,!:;–—-]+\s*",
    r"^\s*hi\s*[,!:;–—-]+\s*",

    # Customer-service wrappers
    r"^\s*hi\s+customer\s+service\s*[-–—,:;]+\s*",
    r"^\s*hello\s+customer\s+service\s*[-–—,:;]+\s*",
    r"^\s*dear\s+customer\s+service\s*[-–—,:;]+\s*",

    # Complaint-template wrappers
    r"^\s*writing\s+to\s+complain\s*[-–—,:;]+\s*",
    r"^\s*i\s+am\s+writing\s+to\s+complain\s*[-–—,:;]+\s*",
    r"^\s*i['’]?m\s+writing\s+to\s+complain\s*[-–—,:;]+\s*",
    r"^\s*i\s+would\s+like\s+to\s+complain\s*[-–—,:;]+\s*",

    # Other artificial wrappers we can see in the data
    r"^\s*flagging\s+an\s+issue\s*[-–—,:;]+\s*",
    r"^\s*reporting\s+an\s+issue\s*[-–—,:;]+\s*",
    r"^\s*raising\s+an\s+issue\s*[-–—,:;]+\s*",

    r"^\s*i\s+am\s+contacting\s+you\s+because\s*[-–—,:;]+\s*",
    r"^\s*i['’]?m\s+contacting\s+you\s+because\s*[-–—,:;]+\s*",
]


SUFFIX_PATTERNS = [
    r"\s*please\s+advise\.?\s*$",
    r"\s*please\s+help\.?\s*$",
    r"\s*please\s+assist\.?\s*$",
    r"\s*kindly\s+advise\.?\s*$",
    r"\s*can\s+you\s+help\??\s*$",
    r"\s*could\s+you\s+help\??\s*$",
    r"\s*i\s+need\s+help\.?\s*$",
    r"\s*thank\s+you\.?\s*$",
    r"\s*thanks\.?\s*$",
]


def basic_clean(text):

    text = str(text)

    text = unicodedata.normalize("NFKC", text)

    text = text.replace("’", "'")
    text = text.replace("‘", "'")
    text = text.replace("“", '"')
    text = text.replace("”", '"')
    text = text.replace("–", "-")
    text = text.replace("—", "-")

    text = re.sub(r"\s+", " ", text)

    return text.strip()


def extract_core(text):

    text = basic_clean(text)

    # Repeat in case there is more than one wrapper
    for _ in range(4):

        old_text = text

        for pattern in PREFIX_PATTERNS:
            text = re.sub(
                pattern,
                "",
                text,
                flags=re.IGNORECASE
            )

        for pattern in SUFFIX_PATTERNS:
            text = re.sub(
                pattern,
                "",
                text,
                flags=re.IGNORECASE
            )

        text = text.strip(" \t\n\r-,:;.")

        if text == old_text:
            break

    text = re.sub(r"\s+", " ", text)

    return text.strip()


def make_core_key(text):

    text = extract_core(text).lower()

    # punctuation shouldn't prevent duplicate matching
    text = re.sub(r"[^\w\s]", " ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()


# Recreate columns from ORIGINAL text
train_df["text"] = train_df["text"].fillna("").astype(str)
test_df["text"] = test_df["text"].fillna("").astype(str)

train_df["core_text"] = train_df["text"].apply(extract_core)
test_df["core_text"] = test_df["text"].apply(extract_core)

train_df["core_key"] = train_df["text"].apply(make_core_key)
test_df["core_key"] = test_df["text"].apply(make_core_key)


print("Training rows:", len(train_df))
print("Unique original text:", train_df["text"].nunique())
print("Unique complaint cores:", train_df["core_key"].nunique())

Training rows: 380
Unique original text: 380
Unique complaint cores: 86


In [6]:
pd.set_option("display.max_colwidth", 200)

display(
    train_df[
        ["text", "core_text", "Category"]
    ].head(20)
)

,text,core_text,Category
0,Flagging an issue: Username change broke access to digital subscriptions tied to old email. Please advise.,Username change broke access to digital subscriptions tied to old email,account_access
1,Writing to complain: Blender motor smells like burning plastic during low-speed blending. Please advise.,Blender motor smells like burning plastic during low-speed blending,product_defect
2,"Hello, Courier left perishable groceries in the sun despite shade instructions.",Courier left perishable groceries in the sun despite shade instructions,delivery_shipping
3,Hi support — Tablet touchscreen registers ghost touches along the right edge.,Tablet touchscreen registers ghost touches along the right edge,product_defect
4,Hi support — Authorized service charged labor despite comprehensive coverage plan.,Authorized service charged labor despite comprehensive coverage plan,warranty_repair
5,Writing to complain: Locked out after suspicious login attempts I did not recognize. Please advise.,Locked out after suspicious login attempts I did not recognize,account_access
6,Hi support — Cancellation effective next year according to agent though terms say monthly.,Cancellation effective next year according to agent though terms say monthly,subscription_cancel
7,Writing to complain: Can I buy spare filters in bulk through the corporate portal? Please advise.,Can I buy spare filters in bulk through the corporate portal?,general_inquiry
8,Defective oven element failed again after warranty repair last quarter.,Defective oven element failed again after warranty repair last quarter,warranty_repair
9,Tablet touchscreen registers ghost touches along the right edge.,Tablet touchscreen registers ghost touches along the right edge,product_defect


In [7]:
train_core_set = set(train_df["core_key"])

test_overlap = test_df["core_key"].isin(train_core_set)

print("=" * 60)
print("TRAIN ↔ TEST CORE OVERLAP")
print("=" * 60)

print(
    f"{test_overlap.sum()} / {len(test_df)} test rows overlap"
)

print(
    f"{test_overlap.mean() * 100:.2f}% of test data"
)

print("\nUnique test cores:", test_df["core_key"].nunique())

print(
    "Unique test cores found in train:",
    test_df.loc[test_overlap, "core_key"].nunique()
)

TRAIN ↔ TEST CORE OVERLAP
0 / 160 test rows overlap
0.00% of test data

Unique test cores: 37
Unique test cores found in train: 0


In [8]:
conflict_table = (
    train_df
    .groupby("core_key")["Category"]
    .nunique()
)

conflicts = conflict_table[
    conflict_table > 1
]

print(
    "Core complaints with conflicting labels:",
    len(conflicts)
)

if len(conflicts) > 0:

    print("\nConflicting examples:")

    for core in conflicts.index[:10]:

        print("\nCORE:", core)

        display(
            train_df[
                train_df["core_key"] == core
            ][
                ["text", "Category"]
            ]
        )

Core complaints with conflicting labels: 0


In [9]:
def majority_label(series):
    return series.value_counts().index[0]


unique_train = (
    train_df
    .groupby("core_key", as_index=False)
    .agg(
        core_text=("core_text", "first"),
        Category=("Category", majority_label),
        duplicate_count=("Category", "size"),
        n_labels=("Category", "nunique"),
    )
)

unique_train["label"] = (
    unique_train["Category"]
    .map(label2id)
)


print("Original rows:", len(train_df))
print("Actual unique complaints:", len(unique_train))

print("\nUnique complaints per category:")

display(
    unique_train["Category"]
    .value_counts()
    .reindex(LABELS)
)

Original rows: 380
Actual unique complaints: 86

Unique complaints per category:


Category
billing                12
product_defect          7
delivery_shipping       9
refund_return           9
customer_service        6
account_access          8
fraud_unauthorized     10
warranty_repair         9
subscription_cancel     9
general_inquiry         7
Name: count, dtype: int64

In [10]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

X = unique_train["core_text"].values
y = unique_train["label"].values

N_FOLDS = 3

skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED
)

folds = list(
    skf.split(X, y)
)

for fold, (train_idx, valid_idx) in enumerate(folds):

    print(
        f"Fold {fold+1}: "
        f"Train={len(train_idx)}, "
        f"Validation={len(valid_idx)}"
    )

Fold 1: Train=57, Validation=29
Fold 2: Train=57, Validation=29
Fold 3: Train=58, Validation=28


In [11]:
from sklearn.pipeline import FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression


def build_tfidf():

    word = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        lowercase=True,
        strip_accents="unicode",
        sublinear_tf=True,
        min_df=1,
        max_features=30000
    )

    char = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 6),
        lowercase=True,
        sublinear_tf=True,
        min_df=1,
        max_features=50000
    )

    return FeatureUnion([
        ("word", word),
        ("char", char)
    ])

In [12]:
C_VALUES = [
    0.25,
    0.5,
    1,
    2,
    4,
    8,
    16,
    32
]

best_tfidf_score = -1
best_tfidf_C = None
best_tfidf_oof = None


for C in C_VALUES:

    oof_probs = np.zeros(
        (len(unique_train), len(LABELS))
    )

    for train_idx, valid_idx in folds:

        vectorizer = build_tfidf()

        X_train_vec = vectorizer.fit_transform(
            X[train_idx]
        )

        X_valid_vec = vectorizer.transform(
            X[valid_idx]
        )


        clf = LogisticRegression(
            C=C,
            class_weight="balanced",
            max_iter=5000,
            random_state=SEED
        )

        clf.fit(
            X_train_vec,
            y[train_idx]
        )


        probabilities = clf.predict_proba(
            X_valid_vec
        )


        fold_probs = np.zeros(
            (
                len(valid_idx),
                len(LABELS)
            )
        )

        fold_probs[:, clf.classes_] = probabilities

        oof_probs[valid_idx] = fold_probs


    predictions = oof_probs.argmax(axis=1)

    score = f1_score(
        y,
        predictions,
        average="macro"
    )


    print(
        f"C={C:<6}  Macro F1={score:.4f}"
    )


    if score > best_tfidf_score:

        best_tfidf_score = score
        best_tfidf_C = C
        best_tfidf_oof = oof_probs.copy()


print("\nBEST TF-IDF")
print("C =", best_tfidf_C)
print("Macro F1 =", best_tfidf_score)

C=0.25    Macro F1=0.4695
C=0.5     Macro F1=0.4695
C=1       Macro F1=0.4382
C=2       Macro F1=0.4386
C=4       Macro F1=0.4381
C=8       Macro F1=0.4381
C=16      Macro F1=0.4383
C=32      Macro F1=0.4354

BEST TF-IDF
C = 0.25
Macro F1 = 0.46949758085052207


In [13]:
%pip install -q sentence-transformers sentencepiece

Note: you may need to restart the kernel to use updated packages.


In [14]:
from sentence_transformers import SentenceTransformer

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

embedding_model = SentenceTransformer(
    "sentence-transformers/all-mpnet-base-v2",
    device=device
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [15]:
train_embeddings = embedding_model.encode(
    unique_train["core_text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

test_embeddings = embedding_model.encode(
    test_df["core_text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print(train_embeddings.shape)
print(test_embeddings.shape)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

(86, 768)
(160, 768)


In [16]:
EMBED_C_VALUES = [
    0.05,
    0.1,
    0.3,
    1,
    3,
    10,
    30,
    100
]

best_embed_score = -1
best_embed_C = None
best_embed_oof = None


for C in EMBED_C_VALUES:

    oof_probs = np.zeros(
        (len(unique_train), len(LABELS))
    )


    for train_idx, valid_idx in folds:

        clf = LogisticRegression(
            C=C,
            class_weight="balanced",
            max_iter=5000,
            random_state=SEED
        )


        clf.fit(
            train_embeddings[train_idx],
            y[train_idx]
        )


        probabilities = clf.predict_proba(
            train_embeddings[valid_idx]
        )


        fold_probs = np.zeros(
            (
                len(valid_idx),
                len(LABELS)
            )
        )

        fold_probs[:, clf.classes_] = probabilities

        oof_probs[valid_idx] = fold_probs


    predictions = oof_probs.argmax(axis=1)

    score = f1_score(
        y,
        predictions,
        average="macro"
    )


    print(
        f"C={C:<6} Macro F1={score:.4f}"
    )


    if score > best_embed_score:

        best_embed_score = score
        best_embed_C = C
        best_embed_oof = oof_probs.copy()


print("\nBEST MPNet")
print("C =", best_embed_C)
print("Macro F1 =", best_embed_score)

C=0.05   Macro F1=0.7073
C=0.1    Macro F1=0.7073
C=0.3    Macro F1=0.6999
C=1      Macro F1=0.7166
C=3      Macro F1=0.7141
C=10     Macro F1=0.7556
C=30     Macro F1=0.7508
C=100    Macro F1=0.7471

BEST MPNet
C = 10
Macro F1 = 0.7556255622751787


In [17]:
best_blend_score = -1
best_alpha = None


for alpha in np.arange(
    0,
    1.01,
    0.05
):

    probs = (
        alpha * best_embed_oof
        +
        (1 - alpha) * best_tfidf_oof
    )


    predictions = probs.argmax(axis=1)


    score = f1_score(
        y,
        predictions,
        average="macro"
    )


    print(
        f"MPNet weight={alpha:.2f} "
        f"TFIDF weight={1-alpha:.2f} "
        f"F1={score:.4f}"
    )


    if score > best_blend_score:

        best_blend_score = score
        best_alpha = alpha


print("\nBEST CURRENT ENSEMBLE")
print("Macro F1:", best_blend_score)
print("MPNet weight:", best_alpha)
print("TF-IDF weight:", 1 - best_alpha)

MPNet weight=0.00 TFIDF weight=1.00 F1=0.4695
MPNet weight=0.05 TFIDF weight=0.95 F1=0.6325
MPNet weight=0.10 TFIDF weight=0.90 F1=0.6986
MPNet weight=0.15 TFIDF weight=0.85 F1=0.7516
MPNet weight=0.20 TFIDF weight=0.80 F1=0.7516
MPNet weight=0.25 TFIDF weight=0.75 F1=0.7417
MPNet weight=0.30 TFIDF weight=0.70 F1=0.7459
MPNet weight=0.35 TFIDF weight=0.65 F1=0.7459
MPNet weight=0.40 TFIDF weight=0.60 F1=0.7459
MPNet weight=0.45 TFIDF weight=0.55 F1=0.7459
MPNet weight=0.50 TFIDF weight=0.50 F1=0.7556
MPNet weight=0.55 TFIDF weight=0.45 F1=0.7556
MPNet weight=0.60 TFIDF weight=0.40 F1=0.7556
MPNet weight=0.65 TFIDF weight=0.35 F1=0.7556
MPNet weight=0.70 TFIDF weight=0.30 F1=0.7556
MPNet weight=0.75 TFIDF weight=0.25 F1=0.7556
MPNet weight=0.80 TFIDF weight=0.20 F1=0.7556
MPNet weight=0.85 TFIDF weight=0.15 F1=0.7556
MPNet weight=0.90 TFIDF weight=0.10 F1=0.7556
MPNet weight=0.95 TFIDF weight=0.05 F1=0.7556
MPNet weight=1.00 TFIDF weight=0.00 F1=0.7556

BEST CURRENT ENSEMBLE
Macro F1: 0

In [18]:
final_vectorizer = build_tfidf()

X_train_tfidf = final_vectorizer.fit_transform(
    unique_train["core_text"]
)

X_test_tfidf = final_vectorizer.transform(
    test_df["core_text"]
)

final_tfidf_model = LogisticRegression(
    C=best_tfidf_C,
    class_weight="balanced",
    max_iter=5000,
    random_state=SEED
)

final_tfidf_model.fit(
    X_train_tfidf,
    y
)

raw_probs = final_tfidf_model.predict_proba(
    X_test_tfidf
)

tfidf_test_probs = np.zeros(
    (len(test_df), len(LABELS))
)

tfidf_test_probs[
    :,
    final_tfidf_model.classes_
] = raw_probs

print("Final TF-IDF trained.")

Final TF-IDF trained.


In [19]:
final_mpnet_model = LogisticRegression(
    C=best_embed_C,
    class_weight="balanced",
    max_iter=5000,
    random_state=SEED
)

final_mpnet_model.fit(
    train_embeddings,
    y
)

raw_probs = final_mpnet_model.predict_proba(
    test_embeddings
)

mpnet_test_probs = np.zeros(
    (len(test_df), len(LABELS))
)

mpnet_test_probs[
    :,
    final_mpnet_model.classes_
] = raw_probs

print("Final MPNet trained.")

Final MPNet trained.


In [20]:
# 50/50 ensemble
final_probs = (
    0.50 * mpnet_test_probs
    +
    0.50 * tfidf_test_probs
)

predicted_ids = np.argmax(
    final_probs,
    axis=1
)

final_labels = np.array([
    id2label[i]
    for i in predicted_ids
])

# Create submission
submission = pd.DataFrame({
    "ComplaintId": test_df["ComplaintId"],
    "Category": final_labels
})

# Checks
assert len(submission) == 160
assert list(submission.columns) == ["ComplaintId", "Category"]
assert submission["Category"].isin(LABELS).all()
assert submission["ComplaintId"].equals(test_df["ComplaintId"])
assert submission["Category"].notna().all()

# Save
submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

print("✅ Submission ready")
print("/kaggle/working/submission.csv")

print("\nPrediction distribution:")
print(submission["Category"].value_counts())

display(submission.head(20))

✅ Submission ready
/kaggle/working/submission.csv

Prediction distribution:
Category
billing                31
subscription_cancel    25
warranty_repair        20
delivery_shipping      20
fraud_unauthorized     15
refund_return          15
product_defect         15
customer_service       10
general_inquiry         5
account_access          4
Name: count, dtype: int64


,ComplaintId,Category
0,1001,account_access
1,1002,subscription_cancel
2,1003,billing
3,1004,delivery_shipping
4,1005,billing
5,1006,warranty_repair
6,1007,billing
7,1008,customer_service
8,1009,billing
9,1010,delivery_shipping
